In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

import os
import sys

python_exe = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

spark = (
    SparkSession.builder
    .appName("EEG_Schizoprenia")
    .master("local[*]")
    .config("spark.pyspark.python", python_exe)
    .config("spark.pyspark.driver.python", python_exe)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark started:", spark.version)
print("Python:", sc.pythonExec)

## Loading the data

In [ ]:
df = spark.read.parquet("data/processed/preprocessed_final_modeling_table")
df.show()
df.printSchema()

## Splitting the Dataframe to Train and Test

In [ ]:
from pyspark.sql import functions as F

subject_df = df.groupBy("subject").agg(F.first("label").alias("label"))

# split by subjects to avoid data leakage
train_subjects = subject_df.sampleBy("label", fractions={0.0: 0.8, 1.0: 0.8}, seed=4202)
train_df = df.join(train_subjects.select("subject"), on="subject", how="inner")

test_subjects = subject_df.join(train_subjects.select("subject"), on="subject", how="left_anti") # not in train_subjects
test_df = df.join(test_subjects.select("subject"), on="subject", how="inner")

In [ ]:
train_subject_list = [int(row["subject"]) for row in train_subjects.orderBy("subject").select("subject").collect()]
test_subject_list = [int(row["subject"]) for row in test_subjects.orderBy("subject").select("subject").collect()]

print("Train Subjects:", train_subject_list)
print("Test Subjects:", test_subject_list)

# confirm no overlap of subjects
print("Number of participant that exist in both train and test:", len(set(train_subject_list) & set(test_subject_list)))


## Model Training Evaluation 

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

def evaluate_predictions(predictions):
    
    # AUC-ROC: only for binary classification
    binary_evaluator = BinaryClassificationEvaluator( labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    # Accuracy / Precision / Recall / F1
    multi_evaluator = MulticlassClassificationEvaluator( labelCol="label", predictionCol="prediction")
    
    metrics = {}

    metrics["auc_roc"] = binary_evaluator.evaluate(predictions)
    print(f"\t Area Under ROC: {metrics['auc_roc']:.4f}")

    metrics["accuracy"] = multi_evaluator.setMetricName("accuracy").evaluate(predictions)
    print(f"\t Accuracy: {metrics['accuracy']:.4f}")

    metrics["precision"] = multi_evaluator.setMetricName("weightedPrecision").evaluate(predictions)
    print(f"\t Precision: {metrics['precision']:.4f}")

    metrics["recall"] = multi_evaluator.setMetricName("weightedRecall").evaluate(predictions)
    print(f"\t Recall: {metrics['recall']:.4f}")

    metrics["f1"] = multi_evaluator.setMetricName("f1").evaluate(predictions)
    print(f"\t F1 Score: {metrics['f1']:.4f}")

    return metrics

In [ ]:
from pyspark.ml.classification import LogisticRegression

lrModel = (
    LogisticRegression(
        labelCol="label",
        featuresCol="features",
        elasticNetParam=0.5
    )
)

lrFitted = lrModel.fit(train_df)
lrPredictions = lrFitted.transform(test_df)

lr_metrics = evaluate_predictions(lrPredictions)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

dtModel = (
    DecisionTreeClassifier(
        labelCol="label",
        featuresCol="features",
        seed=4202
    )
)

dtFitted = dtModel.fit(train_df)
dtPredictions = dtFitted.transform(test_df)

dt_metrics = evaluate_predictions(dtPredictions)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rfModel = (
    RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        seed=4202
    )
)

rfFitted = rfModel.fit(train_df)
rfPredictions = rfFitted.transform(test_df)

rf_metrics = evaluate_predictions(rfPredictions)

more complex tree models are worse than a simple logistic so ill probably not try xgboost or lightbgm

In [ ]:
from pyspark.ml.classification import LinearSVC

svmModel = (
    LinearSVC(
        labelCol="label",
        featuresCol="features",
        maxIter=100,
        regParam=0.1
    )
)

svmFitted = svmModel.fit(train_df)
svmPredictions = svmFitted.transform(test_df)

svm_metrics = evaluate_predictions(svmPredictions)

In [ ]:
from pyspark.ml.classification import NaiveBayes

nbModel = (
    NaiveBayes(
        labelCol="label",
        featuresCol="features",
        modelType="gaussian"   # cuz continuous
    )
)

nbFitted = nbModel.fit(train_df)
nbPredictions = nbFitted.transform(test_df)

nb_metrics = evaluate_predictions(nbPredictions)

In [ ]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

input_size = len(train_df.select("features").first()[0])

mlpModel = (
    MultilayerPerceptronClassifier(
        labelCol="label",
        featuresCol="features",
        layers=[input_size, 64, 32, 2],
        seed=4202,
        maxIter=100
    )
)

mlpFitted = mlpModel.fit(train_df)
mlpPredictions = mlpFitted.transform(test_df)

mlp_metrics = evaluate_predictions(mlpPredictions)

# I tried a deeper longer model with layers=[input_size, 512, 512, 126, 64, 2]
# but the preformance was worse:
#    Area Under ROC: 0.3537
# 	 Accuracy: 0.3849
# 	 Precision: 0.3817
# 	 Recall: 0.3849
# 	 F1 Score: 0.3832

poor results for all models I want to the training metrics

In [ ]:
models = [
    ("Logistic Regression", lrFitted, lrPredictions),
    ("Decision Tree", dtFitted, dtPredictions),
    ("Random Forest", rfFitted, rfPredictions),
    ("Linear SVC", svmFitted, svmPredictions),
    ("Naive Bayes", nbFitted, nbPredictions),
    ("Multilayer Perceptron", mlpFitted, mlpPredictions),
]

for name, model, test_preds in models:
    print(f"====== {name} ======")
    print("Train Metrics:")
    evaluate_predictions(model.transform(train_df))
    print("Test Metrics:")
    evaluate_predictions(test_preds)
    print()

The models are overfitting and not generalizing well enough with very high train ROC but bad test ROC. Except for the Naive Bayes model, where it's not learning at all. It's not learning at all. The accuracy is very close to a coin flip

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

# split features vector into columns
from pyspark.ml.functions import vector_to_array
df = df.withColumn("features_array", vector_to_array("features"))

num_features = len(df.select("features_array").first()[0])
feature_cols = [f"f_{i}" for i in range(num_features)]

df = df.select(
    "subject", "condition", "label",
    *[F.col("features_array")[i].alias(feature_cols[i]) for i in range(num_features)]
)

# aggregate per (subject, condition)
agg_exprs = [F.mean(c).alias(c) for c in feature_cols]
subject_cond_df = df.groupBy("subject", "condition", "label").agg(*agg_exprs)

# reassemble features
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
subject_cond_df = assembler.transform(subject_cond_df)

# scale
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)
scalerModel = scaler.fit(subject_cond_df)
subject_cond_df = scalerModel.transform(subject_cond_df).select("subject", "condition", "label", "features")

# subject-level split (no leakage)
subject_df = subject_cond_df.select("subject", "label").distinct()

train_subjects = subject_df.sampleBy("label", fractions={0.0: 0.8, 1.0: 0.8}, seed=42)
test_subjects = subject_df.join(train_subjects.select("subject"), on="subject", how="left_anti")

train_df = subject_cond_df.join(train_subjects.select("subject"), on="subject", how="inner")
test_df = subject_cond_df.join(test_subjects.select("subject"), on="subject", how="inner")

# logistic regression
lrModel = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    elasticNetParam=0.5,
    regParam=0.1
)

lrFitted = lrModel.fit(train_df)
lrPredictions = lrFitted.transform(test_df)
print(f"====== {LogisticRegression} ======")
lr_metrics = evaluate_predictions(lrPredictions)


input_size = len(train_df.select("features").first()[0])
mlpModel = (
    MultilayerPerceptronClassifier(
        labelCol="label",
        featuresCol="features",
        layers=[input_size, 512, 512, 126, 64, 2],
        seed=4202,
        maxIter=100
    )
)

mlpFitted = mlpModel.fit(train_df)
mlpPredictions = mlpFitted.transform(test_df)
print(f"====== {MultilayerPerceptronClassifier} ======")
mlp_metrics = evaluate_predictions(mlpPredictions)


In [ ]:
# spark.stop()